# Statistické testování hypotéz

IZV cast3 projektu\
Autor: David Kvacek (xkvace00@stud.fit.vutbr.cz)

Tento Python notebook ověřuje následující dvě hypotézy:
1. *„Na silnicích první třídy byly nehody s následky na zdraví se stejnou pravděpodobností jako na dálnicích.“*
2. *„Škoda při nehodách trolejbusů je nižší, než při nehodách autobusů a tato odchylka je statisticky významná.“*

Pro analýzu jsou použity soubory `accidents.pkl.gz` a `vehicles.pkl.gz`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, shapiro

# load the data from the pickle files
df_accidents = pd.read_pickle("accidents.pkl.gz")
df_vehicles = pd.read_pickle("vehicles.pkl.gz")

### Hypotéza 1

- **Hypotéza**: *„Na silnicích první třídy byly nehody s následky na zdraví se stejnou pravděpodobností jako na dálnicích.“*
- **Metoda**: Pro ověření této hypotézy je použit $\chi^2$ test nezávislosti.

Proměnné:
- **Položka** `p9`: Následky na zdraví (1 = s následky na životě, 2 = pouze s hmotnou škodou).
- **Položka** `p36`: Typ komunikace (1 = dálnice, 2 = silnice první třídy).

Následně je analyzována četnost těchto kategorií.

Nejprve je nutné vybrat relevantní data a připravit je pro analýzu.\
V tomto případě odstranění položek s chybějícími hodnotami a přidání sloupce `injury` pro identifikaci nehod s následky na zdraví.

In [2]:
# choose the relevant columns, drop rows with missing values and add injury column
accident = df_accidents[["p9", "p36"]].dropna()
accident = accident[accident["p36"].isin([0, 1])]
accident["injury"] = (accident["p9"] == 1).astype(int)

Dalším krokem je vytvoření kontingenční tabulky.

In [3]:
# create and print the contingency table for the chi2 test
contingency_table = pd.crosstab(accident["p36"], accident["injury"])
print("\nKontingenční tabulka:\n")
print(contingency_table)


Kontingenční tabulka:

injury      0     1
p36                
0        6674  1247
1       14773  7059


Nyní je možné provést $\chi^2$ test nezávislosti.\
Pomocí metody `chi2_contingency` z knihovny `scipy.stats` jsou získány hodnoty $\chi^2$, $p$-hodnota, stupně volnosti a očekávané hodnoty.

In [4]:
# perform the chi2 test
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"\nchi2: {chi2}\np-hodnota: {p}\nstupně volnosti: {dof}")


chi2: 794.1532980454858
p-hodnota: 1.0075002730159726e-174
stupně volnosti: 1


Na závěr je výsledek testu vyhodnocen a je zodpovězena otázka, zda je možné hypotézu zamítnout, nebo ne.

In [5]:
# print the results of the chi2 test and the conclusion
if p < 0.05:
    print("Hypotéza JE ZAMÍTNUTA: existuje statisticky VÝZNAMNÝ rozdíl mezi nehodami s následky na zdraví na silnicích první třídy a na dálnicích.")

else:
    print("Hypotéza NENÍ ZAMÍTNUTA: existuje statisticky NEVÝZNAMNÝ rozdíl mezi nehodami s následky na zdraví na silnicích první třídy a na dálnicích.")

Hypotéza JE ZAMÍTNUTA: existuje statisticky VÝZNAMNÝ rozdíl mezi nehodami s následky na zdraví na silnicích první třídy a na dálnicích.


### Hypotéza 2

- **Hypotéza**: *„Škoda při nehodách trolejbusů je nižší, než při nehodách autobusů a tato odchylka je statisticky významná.“*
- **Metoda**:
    - Nejprve je provedena kontrola normality dat.
    - Pokud jsou data normální, je použit t-test.
    - V opačném případě je použit Mann-Whitney U test.

Proměnné:
- **Položka** `p14`: Celková hmotná škoda v Kč.
- **Položka** `p44`: Druh vozidla (8 = autobus, 11 = trolejbus).

Nejprve je nutné vybrat relevantní data a připravit je pro analýzu.\
V tomto případě odstranění položek s chybějícími hodnotami, přejmenování sloupce `p14*100` na `p14` a aktualizace hodnot v tomto sloupci.

In [6]:
# choose the relevant columns, drop rows with missing values and rename and actualize the p14 column values
vehicles = df_accidents.merge(df_vehicles, on="p1")
vehicles = vehicles[["p14*100", "p44"]].dropna()
vehicles.rename(columns={"p14*100": "p14"}, inplace=True)
vehicles["p14"] = vehicles["p14"] / 100

Dalším krokem je extrakce dat pro autobusy a trolejbusy.

In [7]:
# extract the relevant data for specific types of vehicles
bus = vehicles[vehicles["p44"] == 8]["p14"]
trolleybus = vehicles[vehicles["p44"] == 11]["p14"]

Následně je provedena kontrola normality dat a na základě výsledku je vybrána metoda pro testování hypotézy.\
Pokud jsou data normální, je použit t-test, jinak je použit Mann-Whitney U test.

In [8]:
# use the appropriate statistical test and print the results
if shapiro(bus).pvalue > 0.05 and shapiro(trolleybus).pvalue > 0.05:
    # if the data is normally distributed, use the t-test
    stat, p = ttest_ind(bus, trolleybus, equal_var=False)
    test = "t-test"

else:
    # if the data is not normally distributed, use the Mann-Whitney U test
    stat, p = mannwhitneyu(bus, trolleybus, alternative="greater")
    test = "Mann-Whitney U test"

print(f"test: {test}\nstatistika: {stat}\np-hodnota: {p}")

test: Mann-Whitney U test
statistika: 787252.5
p-hodnota: 1.2358248015601533e-13


Na závěr je výsledek testu vyhodnocen a je zodpovězena otázka, zda je možné hypotézu zamítnout, nebo ne.

In [9]:
# print the results of the appropriate statistical test and the conclusion
if p < 0.05:
    if bus.median() > trolleybus.median():
        print("Hypotéza NENÍ ZAMÍTNUTA: škoda při nehodách trolejbusů JE NIŽŠÍ, než při nehodách autobusů a tato odchylka je statisticky VÝZNAMNÁ.")

    else:
        print("Hypotéza JE ZAMÍTNUTA: škoda při nehodách trolejbusů NENÍ NIŽŠÍ, než při nehodách autobusů a tato odchylka je statisticky VÝZNAMNÁ.")

else:
    print("Hypotéza JE ZAMÍTNUTA: existuje statisticky NEVÝZNAMNÁ odchylka mezi škodami při nehodách autobusů a trolejbusů.")

Hypotéza NENÍ ZAMÍTNUTA: škoda při nehodách trolejbusů JE NIŽŠÍ, než při nehodách autobusů a tato odchylka je statisticky VÝZNAMNÁ.
